# What Helps with Abortion Recovery?

## Abstract

We analyzed treatment sentiment reports from r/abortion to identify what substances and remedies users report as helpful during post-abortion recovery. After filtering causal-context contamination (birth control, Plan B, medical/surgical abortion procedures, abortion pills, misoprostol, mifepristone, and IUD -- which appear in the data because they caused the situation or are the procedure itself, not recovery aids) and generic non-actionable terms (medication, treatment, pill, drug, pain medication, nausea medicine), we ranked the remaining treatments by user count, positive outcome rate, and statistical significance. Recommendations are tiered by evidence strength (Strong / Moderate / Preliminary) so both patients and researchers can gauge confidence. These findings reflect self-reported community data, not clinical trials.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats as sp_stats
from scipy.stats import binomtest
from pathlib import Path
from IPython.display import display, HTML

# Resolve project root
_here = Path('.').resolve()
PROJECT_ROOT = _here.parent if not (_here / 'app').exists() and (_here.parent / 'app').exists() else _here
sys.path.insert(0, str(PROJECT_ROOT))

from app.analysis.stats import (
    get_user_sentiment,
    run_binomial_test,
    summarize_drug,
    run_comparison,
    run_kruskal_wallis,
    REPORTING_BIAS_DISCLAIMER,
)

DB_PATH = PROJECT_ROOT / 'abortion.db'
conn = sqlite3.connect(str(DB_PATH))

SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0, "weak": 0.25}
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.style.use("seaborn-v0_8-whitegrid")

# --- Causal-context and generic filters ---
# These terms are excluded from recovery analysis because they reflect the
# REASON users are on r/abortion, not post-procedure recovery treatments.
CAUSAL_CONTEXT_FILTERS = [
    'birth control',
    'plan b',
    'medical abortion',
    'medication abortion',
    'surgical abortion',
    'surgical procedure',
    'abortion pill',
    'abortion pills',
    'misoprostol',          # the prostaglandin used IN the abortion procedure
    'mifepristone',         # the anti-progesterone used IN the abortion procedure
    'intrauterine device',  # IUD — contraceptive, same causal bucket as birth control
]

# Generic non-actionable terms a patient cannot look up or ask a doctor about
GENERIC_FILTERS = [
    'medication',
    'treatment',
    'pill',
    'pills',
    'drug',
    'drugs',
    'medicine',
    'supplement',
    'supplements',
    'therapy',
    'vitamins',
    'pain medication',
    'nausea medicine',
    'anti-nausea medication',
]

# Diagnostic procedures (not treatments)
DIAGNOSTIC_FILTERS = [
    'pregnancy test',
    'ultrasound',
]

ALL_FILTERS = [f.lower() for f in CAUSAL_CONTEXT_FILTERS + GENERIC_FILTERS + DIAGNOSTIC_FILTERS]

## 2. Data Exploration

First we check what data is available: row counts, date range, sentiment distribution, and the top treatments before any filtering.

In [ ]:
import datetime

# Row counts
table_counts = {}
for table in ['users', 'posts', 'treatment', 'treatment_reports', 'user_profiles', 'conditions']:
    try:
        n = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        table_counts[table] = n
    except Exception:
        table_counts[table] = 0

# Date range
lo, hi = conn.execute('SELECT MIN(post_date), MAX(post_date) FROM posts').fetchone()
# Handle both unix timestamp and date string formats
try:
    lo_date = datetime.datetime.fromtimestamp(lo).date() if isinstance(lo, (int, float)) else lo
    hi_date = datetime.datetime.fromtimestamp(hi).date() if isinstance(hi, (int, float)) else hi
except (ValueError, TypeError, OSError):
    lo_date, hi_date = lo, hi

# Calculate months span
try:
    if isinstance(lo_date, str):
        _lo = datetime.datetime.strptime(str(lo_date)[:10], '%Y-%m-%d')
        _hi = datetime.datetime.strptime(str(hi_date)[:10], '%Y-%m-%d')
    else:
        _lo = datetime.datetime.combine(lo_date, datetime.time())
        _hi = datetime.datetime.combine(hi_date, datetime.time())
    n_months = max(1, round((_hi - _lo).days / 30.44))
except Exception:
    n_months = '?'

# Build summary HTML
rows_html = ''.join(f'<tr><td style="text-align:left;padding:4px 12px">{t}</td><td style="text-align:right;padding:4px 12px">{n:,}</td></tr>' for t, n in table_counts.items())

display(HTML(f"""
<h3>Dataset Overview — r/abortion</h3>
<table style="border-collapse:collapse;font-size:14px">
<tr><th style="text-align:left;padding:4px 12px">Table</th><th style="text-align:right;padding:4px 12px">Rows</th></tr>
{rows_html}
</table>
<p style="margin-top:10px"><strong>Data covers:</strong> {lo_date} to {hi_date} ({n_months} months)</p>
"""))

### Sentiment distribution across all treatment reports

Before filtering, this shows the overall balance of positive, mixed, and negative sentiment in the raw data.

In [ ]:
# Load all treatment reports
df_all = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment,
           tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

df_all['score'] = df_all['sentiment'].map(SENTIMENT_SCORE)

# Sentiment pie chart
sent_counts = df_all['sentiment'].value_counts()
pie_colors = {
    'positive': '#2ecc71', 'mixed': '#f39c12', 'neutral': '#bdc3c7',
    'negative': '#e74c3c', 'weak': '#85c1e9'
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sentiment pie
labels = sent_counts.index.tolist()
sizes = sent_counts.values
colors_pie = [pie_colors.get(l, '#bdc3c7') for l in labels]
wedges, texts, autotexts = axes[0].pie(
    sizes, labels=labels, autopct='%1.1f%%', colors=colors_pie,
    startangle=90, textprops={'fontsize': 10}
)
axes[0].set_title(f'Sentiment Distribution (all {len(df_all):,} reports)', fontsize=12)

# Right: top 20 treatments by user count (unfiltered)
top_unfiltered = (
    df_all.groupby('drug')['user_id'].nunique()
    .sort_values(ascending=True)
    .tail(20)
)
bar_colors = ['#e74c3c' if d.lower() in ALL_FILTERS else '#3498db' for d in top_unfiltered.index]
axes[1].barh(range(len(top_unfiltered)), top_unfiltered.values, color=bar_colors, edgecolor='white')
axes[1].set_yticks(range(len(top_unfiltered)))
axes[1].set_yticklabels(top_unfiltered.index, fontsize=9)
axes[1].set_xlabel('Unique users')
axes[1].set_title('Top 20 Treatments by User Count\n(red = excluded from recovery analysis)', fontsize=11)

plt.tight_layout()
plt.show()

## 3. Causal-Context and Generic Filtering

In r/abortion, several high-volume treatments appear not because they *help* with recovery, but because they are the *reason* users are in the community. We exclude these from recovery analysis:

**Causal-context exclusions:**
- **Birth control** (11% positive) -- negative sentiment reflects contraceptive *failure* (why the user needed an abortion), not a failed recovery treatment
- **Plan B** -- same reason: emergency contraception that did not prevent pregnancy
- **Medical abortion / medication abortion / surgical abortion / surgical procedure / abortion pills** -- these are the *procedures themselves*, not recovery treatments. Sentiment reflects the procedure experience, not post-procedure recovery
- **Misoprostol** -- the prostaglandin used *during* the abortion procedure (often called "the second pill"). Sentiment reflects the procedure experience (cramping, bleeding, pain), not recovery
- **Mifepristone** -- the anti-progesterone used to *initiate* the abortion (the "first pill"). Same reasoning as misoprostol
- **Intrauterine device (IUD)** -- a contraceptive device in the same causal bucket as birth control; discussed in context of post-abortion contraception planning, not recovery

**Generic exclusions:** medication, treatment, pill, drug, medicine, supplement, therapy, vitamins, pain medication, nausea medicine, anti-nausea medication -- categories a patient cannot look up or ask their doctor about.

**Diagnostic exclusions:** pregnancy test, ultrasound -- diagnostic procedures, not treatments.

The chart below shows how many reports are removed by each filter.

In [ ]:
# Show what gets filtered and why
filter_impact = []
for f in ALL_FILTERS:
    mask = df_all['drug'].str.lower() == f
    if mask.sum() > 0:
        subset = df_all[mask]
        n_reports = len(subset)
        n_users = subset['user_id'].nunique()
        pct_pos = (subset['score'] > 0).sum() / len(subset) * 100 if len(subset) > 0 else 0
        if f in [x.lower() for x in CAUSAL_CONTEXT_FILTERS]:
            category = 'Causal-context'
        elif f in [x.lower() for x in DIAGNOSTIC_FILTERS]:
            category = 'Diagnostic'
        else:
            category = 'Generic'
        filter_impact.append({
            'Treatment': f,
            'Category': category,
            'Reports': n_reports,
            'Users': n_users,
            '% Positive': round(pct_pos, 1),
        })

filter_df = pd.DataFrame(filter_impact).sort_values('Reports', ascending=False)

if len(filter_df) > 0:
    # Visualization: what we're filtering out
    fig, ax = plt.subplots(figsize=(10, max(3, len(filter_df) * 0.4)))
    y = np.arange(len(filter_df))
    cat_colors = {'Causal-context': '#e74c3c', 'Generic': '#e67e22', 'Diagnostic': '#8e44ad'}
    bar_colors = [cat_colors.get(c, '#999') for c in filter_df['Category']]
    ax.barh(y, filter_df['Reports'].values, color=bar_colors, edgecolor='white', height=0.6)
    ax.set_yticks(y)
    ax.set_yticklabels(filter_df['Treatment'].values, fontsize=10)
    for i, row in enumerate(filter_df.itertuples()):
        ax.text(row.Reports + 0.5, i, f'{row._5:.0f}% pos, n={row.Users}',
                va='center', fontsize=9, color='#555')
    ax.set_xlabel('Number of reports removed')
    ax.set_title('Excluded Treatments by Category', fontsize=12)
    ax.legend(
        handles=[
            plt.Rectangle((0,0),1,1, fc='#e74c3c', label='Causal-context'),
            plt.Rectangle((0,0),1,1, fc='#e67e22', label='Generic'),
            plt.Rectangle((0,0),1,1, fc='#8e44ad', label='Diagnostic'),
        ],
        loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False
    )
    fig.subplots_adjust(bottom=0.18)
    plt.tight_layout()
    plt.show()

    display(HTML(filter_df.to_html(index=False)))
else:
    display(HTML('<p>No filtered treatments found in the data.</p>'))

# Apply filters
df_filtered = df_all[~df_all['drug'].str.lower().isin(ALL_FILTERS)].copy()

n_before = len(df_all)
n_after = len(df_filtered)
display(HTML(f'<p><strong>Filtering:</strong> {n_before:,} reports &rarr; {n_after:,} reports ({n_before - n_after:,} removed, {(n_before - n_after)/n_before*100:.1f}%)</p>'))

## 4. Recovery Treatment Outcomes

With causal-context and generic terms filtered out, we aggregate to the **user level** (one data point per user per drug for statistical independence) and rank treatments by user count and positive outcome rate. A binomial test checks whether each treatment's positive rate significantly differs from 50% (chance).

In [ ]:
# User-level aggregation
user_drug = df_filtered.groupby(['drug', 'user_id']).agg(
    avg_score=('score', 'mean'),
    n_reports=('score', 'count')
).reset_index()

user_drug['outcome'] = user_drug['avg_score'].apply(
    lambda x: 'positive' if x > 0.7 else ('negative' if x < -0.3 else 'mixed/neutral')
)

# Drug summary with binomial tests
drug_counts = user_drug.groupby('drug')['user_id'].nunique().reset_index()
drug_counts.columns = ['drug', 'n_users']
eligible_drugs = drug_counts[drug_counts['n_users'] >= 3].sort_values('n_users', ascending=False)

results = []
for _, row in eligible_drugs.iterrows():
    drug = row['drug']
    users = user_drug[user_drug['drug'] == drug]
    n = len(users)
    n_pos = (users['outcome'] == 'positive').sum()
    n_neg = (users['outcome'] == 'negative').sum()
    n_mix = n - n_pos - n_neg
    test = binomtest(n_pos, n, p=0.5)
    ci = test.proportion_ci(confidence_level=0.95)
    results.append({
        'Treatment': drug,
        'Users': n,
        'Positive': n_pos,
        'Mixed': n_mix,
        'Negative': n_neg,
        '% Positive': round(n_pos / n * 100, 1),
        '95% CI': f'{ci.low*100:.0f}–{ci.high*100:.0f}%',
        'Avg Score': round(users['avg_score'].mean(), 2),
        'p-value': round(test.pvalue, 4),
        'Sig': '*' if test.pvalue < 0.05 else '',
    })

results_df = pd.DataFrame(results)

display(HTML(f'<h3>Recovery Treatments with 3+ User Reports ({len(results_df)} treatments)</h3>'))
display(HTML('<p><em>* = positive rate significantly different from 50%, p &lt; 0.05 (binomial test)</em></p>'))
display(HTML(results_df.head(25).to_html(index=False)))

### Diverging bar chart: recovery treatment outcomes

Each bar represents one treatment. Green extends right (positive outcomes), red extends left (negative), grey in the centre (mixed/neutral). Treatments are ranked by positive outcome rate. Only treatments with 3+ unique user reports are shown.

In [ ]:
# Diverging bar chart for top recovery treatments
plot_df = results_df.head(20).sort_values('% Positive').copy()
n_drugs = len(plot_df)

if n_drugs > 0:
    fig, ax = plt.subplots(figsize=(12, max(5, n_drugs * 0.42)))

    drugs = plot_df['Treatment'].values
    pct_pos = plot_df['% Positive'].values
    pct_neg = (plot_df['Negative'] / plot_df['Users'] * 100).values
    pct_mix = 100 - pct_pos - pct_neg
    y = np.arange(len(drugs))

    # Correct stacking: mixed INNERMOST (adjacent to center), negative OUTERMOST
    ax.barh(y, -pct_mix, height=0.6, color='#d5d8dc',
            label='Mixed/Neutral', edgecolor='white', linewidth=0.5)
    ax.barh(y, -pct_neg, height=0.6, left=-pct_mix,
            color='#e74c3c', label='Negative', edgecolor='white', linewidth=0.5)
    ax.barh(y, pct_pos, height=0.6,
            color='#2ecc71', label='Positive', edgecolor='white', linewidth=0.5)

    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([d.title() for d in drugs], fontsize=10)

    for i, row in enumerate(plot_df.itertuples()):
        ax.text(max(pct_pos) + 3, i, f'n={row.Users}', va='center', fontsize=9, color='gray')
        if row._6 > 15:  # % Positive
            ax.text(row._6 / 2, i, f'{row._6:.0f}%',
                    va='center', ha='center', fontsize=8, color='white', fontweight='bold')

    # Absolute-value x-axis labels
    max_val = max(max(pct_neg + pct_mix) if len(pct_neg) > 0 else 0, max(pct_pos) if len(pct_pos) > 0 else 0)
    tick_range = int(max_val / 20 + 1) * 20
    ticks = list(range(-tick_range, tick_range + 1, 20))
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'{abs(t)}%' for t in ticks])

    ax.set_xlabel('\u2190 % Negative          % Positive \u2192', fontsize=11)
    ax.set_title('Abortion Recovery: Treatment Outcome Distribution\n(r/abortion, user-level aggregation)', fontsize=13)

    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08),
              ncol=3, frameon=False, fontsize=10)
    ax.spines['left'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.subplots_adjust(bottom=0.15)
    plt.tight_layout()
    plt.show()

### Chart interpretation

The diverging bar chart above shows the proportion of users reporting positive, mixed/neutral, and negative outcomes for each recovery treatment. Bars extending further to the right indicate more consistently positive community sentiment. Treatments near the top have the highest positive rates. The `n=` labels on the right show sample size — larger samples give more reliable estimates.

## 5. Precision of Estimates: Forest Plot

The diverging bar chart shows *proportions* but not *uncertainty*. This forest plot shows the mean sentiment score with 95% confidence intervals for each treatment. Narrower intervals = more precise estimates. Treatments where the CI crosses zero have uncertain directionality.

In [ ]:
# Forest plot: mean sentiment +/- 95% CI
forest_drugs = results_df.head(20).sort_values('Avg Score').copy()
n_drugs = len(forest_drugs)

if n_drugs > 0:
    fig, ax = plt.subplots(figsize=(10, max(5, n_drugs * 0.38)))

    forest_data = []
    for _, row in forest_drugs.iterrows():
        drug = row['Treatment']
        scores = user_drug[user_drug['drug'] == drug]['avg_score'].values
        n = len(scores)
        mean = scores.mean()
        if n >= 2:
            se = sp_stats.sem(scores)
            ci = se * sp_stats.t.ppf(0.975, n - 1)
        else:
            ci = 0
        forest_data.append({'drug': drug, 'mean': mean, 'ci': ci, 'n': n})

    means = [d['mean'] for d in forest_data]
    cis = [d['ci'] for d in forest_data]
    ns = [d['n'] for d in forest_data]
    y = np.arange(len(forest_data))
    dot_colors = ['#2ecc71' if m > 0.3 else '#e74c3c' if m < -0.3 else '#95a5a6' for m in means]

    ax.errorbar(means, y, xerr=cis, fmt='none', ecolor='#555555',
                elinewidth=1.2, capsize=3, zorder=1)
    ax.scatter(means, y, c=dot_colors, s=60, zorder=2,
               edgecolors='white', linewidth=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

    ax.set_yticks(y)
    ax.set_yticklabels([d['drug'].title() for d in forest_data], fontsize=10)

    for i, d in enumerate(forest_data):
        ax.text(1.15, i, f'n={d["n"]}', va='center', fontsize=8, color='gray')

    ax.set_xlabel('Mean sentiment score (95% CI)', fontsize=11)
    ax.set_title('Recovery Treatment Sentiment Precision\n(dot = mean, whiskers = 95% CI)', fontsize=13)
    ax.set_xlim(-1.3, 1.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

### Chart interpretation

Each dot represents the mean user-level sentiment score for a treatment. Whiskers show the 95% confidence interval. Treatments plotted in green (mean > 0.3) have clearly positive community sentiment. Grey dots near zero have ambiguous signal. Treatments with wide confidence intervals (few users) should be interpreted cautiously — the true effect could be anywhere in that range.

## 6. Multi-Treatment Comparison (Kruskal-Wallis)

Are there statistically significant differences in sentiment *between* the top recovery treatments? The Kruskal-Wallis test is a non-parametric one-way ANOVA that asks: does at least one treatment differ from the others?

In [ ]:
from scipy.stats import kruskal

# Use top treatments with 5+ users for Kruskal-Wallis
kw_eligible = results_df[results_df['Users'] >= 5].head(8)
kw_drugs = kw_eligible['Treatment'].tolist()

if len(kw_drugs) >= 3:
    kw_groups = [user_drug[user_drug['drug'] == d]['avg_score'].values for d in kw_drugs]
    h_stat, kw_p = kruskal(*kw_groups)

    display(HTML(f"""
    <h3>Kruskal-Wallis Test Across Top {len(kw_drugs)} Recovery Treatments</h3>
    <table style="border-collapse:collapse;font-size:14px">
    <tr><td style="padding:4px 12px"><strong>Treatments:</strong></td><td style="padding:4px 12px">{', '.join(d.title() for d in kw_drugs)}</td></tr>
    <tr><td style="padding:4px 12px"><strong>H statistic:</strong></td><td style="padding:4px 12px">{h_stat:.2f}</td></tr>
    <tr><td style="padding:4px 12px"><strong>p-value:</strong></td><td style="padding:4px 12px">{kw_p:.4f}</td></tr>
    <tr><td style="padding:4px 12px"><strong>Significant:</strong></td><td style="padding:4px 12px">{'Yes \u2014 at least one treatment differs' if kw_p < 0.05 else 'No significant difference across treatments'}</td></tr>
    </table>
    """))

    # Grouped bar chart
    kw_data = user_drug[user_drug['drug'].isin(kw_drugs)].copy()
    counts = (
        kw_data.groupby(['drug', 'outcome'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=['negative', 'mixed/neutral', 'positive'], fill_value=0)
    )
    # Reorder to match kw_drugs order
    counts = counts.reindex([d for d in kw_drugs if d in counts.index])

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(counts))
    w = 0.25
    bar_colors = {'negative': '#e74c3c', 'mixed/neutral': '#95a5a6', 'positive': '#2ecc71'}
    for j, cat in enumerate(['negative', 'mixed/neutral', 'positive']):
        ax.bar(x + (j - 1) * w, counts[cat], width=w,
               color=bar_colors[cat], label=cat.capitalize(), edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels([d.title() for d in counts.index], rotation=20, ha='right', fontsize=10)
    ax.set_ylabel('Number of users')
    ax.set_title(f'Top Recovery Treatments \u2014 Sentiment Distribution (Kruskal-Wallis p={kw_p:.3f})', fontsize=12)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
    fig.subplots_adjust(bottom=0.25)
    plt.tight_layout()
    plt.show()

    if kw_p < 0.05:
        display(HTML('<p><strong>What this means:</strong> There are statistically significant differences in community sentiment across these recovery treatments. Some treatments are rated notably more positively than others.</p>'))
    else:
        display(HTML('<p><strong>What this means:</strong> No statistically significant difference was detected between these treatments. Community sentiment is broadly similar across the top recovery options, though individual rates vary.</p>'))

elif len(kw_drugs) >= 1:
    display(HTML(f'<p>Only {len(kw_drugs)} treatment(s) with 5+ users \u2014 need at least 3 for Kruskal-Wallis. Showing descriptive comparison only.</p>'))
else:
    display(HTML('<p>No treatments with 5+ users. Showing descriptive results only.</p>'))

### Chart interpretation

The grouped bar chart shows the raw count of users in each sentiment category for the top recovery treatments. This complements the percentage-based diverging chart by showing absolute sample sizes, making it easier to see which treatments have the most data behind them.

## 7. Co-occurring Conditions

What conditions or concerns do r/abortion users mention alongside treatment reports? This helps contextualize *why* specific recovery treatments are discussed.

In [ ]:
conditions = pd.read_sql("""
    SELECT condition_name, COUNT(DISTINCT user_id) as n_users
    FROM conditions
    GROUP BY condition_name
    ORDER BY n_users DESC
    LIMIT 15
""", conn)

if len(conditions) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(conditions) * 0.35)))
    ax.barh(range(len(conditions)), conditions['n_users'].values, color='#8e44ad', edgecolor='white')
    ax.set_yticks(range(len(conditions)))
    ax.set_yticklabels(conditions['condition_name'].values, fontsize=10)
    ax.set_xlabel('Number of users')
    ax.set_title('Top Conditions Mentioned in r/abortion', fontsize=12)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    display(HTML('<p>No conditions data available in this database.</p>'))

### Chart interpretation

This chart shows the most frequently mentioned conditions or concerns among users who also reported treatment experiences. It contextualizes the recovery landscape — common concerns (pain, anxiety, cramping, bleeding) drive which treatments are discussed.

## 8. Recommendations

Treatments are tiered by evidence strength based on sample size and statistical significance:

- **Strong evidence:** n ≥ 30 and p < 0.05 (binomial test)
- **Moderate evidence:** n ≥ 15, or p < 0.10
- **Preliminary:** n < 15, not statistically significant

All recommendations are based on community-reported outcomes, not clinical trials.

In [ ]:
# Tier treatments
def assign_tier(row):
    if row['Users'] >= 30 and row['p-value'] < 0.05:
        return 'Strong'
    elif row['Users'] >= 15 or row['p-value'] < 0.10:
        return 'Moderate'
    else:
        return 'Preliminary'

# Only recommend treatments with >50% positive rate
rec_df = results_df[results_df['% Positive'] > 50].copy()
rec_df['Tier'] = rec_df.apply(assign_tier, axis=1)

tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#bdc3c7'}
tier_order = ['Strong', 'Moderate', 'Preliminary']

# Build HTML recommendation table
html_parts = []
for tier in tier_order:
    tier_data = rec_df[rec_df['Tier'] == tier].sort_values('% Positive', ascending=False)
    if len(tier_data) == 0:
        continue
    color = tier_colors[tier]
    html_parts.append(f'<h3 style="color:{color}">{tier} Evidence</h3>')
    for _, row in tier_data.iterrows():
        sig_note = f' (p={row["p-value"]:.3f})' if row['p-value'] < 0.10 else ''
        html_parts.append(
            f'<p><strong>{row["Treatment"].title()}</strong> \u2014 '
            f'{row["% Positive"]:.0f}% positive (n={row["Users"]}, '
            f'95% CI: {row["95% CI"]}){sig_note}</p>'
        )

if len(html_parts) > 0:
    display(HTML('\n'.join(html_parts)))
else:
    display(HTML('<p>No treatments with >50% positive rate found after filtering.</p>'))

# Visual summary: forest plot of recommended treatments, color-coded by tier
if len(rec_df) > 0:
    plot_rec = rec_df.sort_values('% Positive', ascending=True).copy()
    n_rec = len(plot_rec)

    fig, ax = plt.subplots(figsize=(10, max(4, n_rec * 0.38)))
    y = np.arange(n_rec)

    rec_means, rec_cis = [], []
    for _, row in plot_rec.iterrows():
        scores = user_drug[user_drug['drug'] == row['Treatment']]['avg_score'].values
        m = scores.mean()
        rec_means.append(m)
        if len(scores) >= 2:
            se = sp_stats.sem(scores)
            rec_cis.append(se * sp_stats.t.ppf(0.975, len(scores) - 1))
        else:
            rec_cis.append(0)

    rec_colors = [tier_colors[t] for t in plot_rec['Tier']]

    ax.errorbar(rec_means, y, xerr=rec_cis, fmt='none', ecolor='#555',
                elinewidth=1.2, capsize=3, zorder=1)
    ax.scatter(rec_means, y, c=rec_colors, s=80, zorder=2,
               edgecolors='white', linewidth=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

    ax.set_yticks(y)
    ax.set_yticklabels(
        [f'{row.Treatment.title()} (n={row.Users})' for row in plot_rec.itertuples()],
        fontsize=10
    )
    ax.set_xlabel('Mean sentiment score (95% CI)', fontsize=11)
    ax.set_title('Recommended Recovery Treatments by Evidence Tier', fontsize=13)
    ax.set_xlim(-1.0, 1.4)

    # Tier legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=tier_colors[t],
               markersize=10, label=f'{t} evidence')
        for t in tier_order if t in rec_df['Tier'].values
    ]
    ax.legend(handles=legend_elements, loc='upper center',
              bbox_to_anchor=(0.5, -0.10), ncol=3, frameon=False, fontsize=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.subplots_adjust(bottom=0.15)
    plt.tight_layout()
    plt.show()

### Chart interpretation

This forest plot shows all recommended treatments color-coded by evidence tier. Green dots = strong evidence (large sample, statistically significant). Orange dots = moderate evidence. Grey dots = preliminary (small sample, not yet significant). The wider the confidence interval, the less certain we are about the true effect. Treatments further to the right have more consistently positive community sentiment.

## 9. Summary

In [ ]:
# Build dynamic summary
n_total_reports = len(df_all)
n_filtered_reports = len(df_filtered)
n_unique_drugs_filtered = df_filtered['drug'].nunique()
n_unique_users = df_filtered['user_id'].nunique()

# Top treatments by positive rate (with 3+ users)
top_positive = results_df.head(5)

summary_parts = [
    '<h3>Key Findings</h3>',
    f'<ul>',
    f'<li><strong>{n_total_reports:,} treatment reports</strong> across {df_all["drug"].nunique()} substances from r/abortion</li>',
    f'<li>After filtering causal-context and generic terms: <strong>{n_filtered_reports:,} reports</strong> across <strong>{n_unique_drugs_filtered} recovery-specific treatments</strong> from <strong>{n_unique_users} unique users</strong></li>',
]

if len(top_positive) > 0:
    top_list = ', '.join(
        f'{r.Treatment.title()} ({r._6:.0f}% positive, n={r.Users})'
        for r in top_positive.itertuples()
    )
    summary_parts.append(f'<li><strong>Top recovery treatments:</strong> {top_list}</li>')

summary_parts.append('</ul>')

# Causal-context note with specific reasons
causal_reasons = {
    'birth control': 'contraceptive failure (why the user needed an abortion)',
    'plan b': 'emergency contraception that failed to prevent pregnancy',
    'medical abortion': 'the procedure itself, not a recovery treatment',
    'medication abortion': 'the procedure itself (alternate name for medical abortion)',
    'surgical abortion': 'the procedure itself, not a recovery treatment',
    'surgical procedure': 'the procedure itself (alternate name for surgical abortion)',
    'abortion pill': 'the procedure itself (colloquial name for medical abortion)',
    'abortion pills': 'the procedure itself (colloquial name for medical abortion)',
    'misoprostol': 'the prostaglandin used during the abortion procedure',
    'mifepristone': 'the anti-progesterone that initiates the abortion',
    'intrauterine device': 'a contraceptive (IUD), not a recovery treatment',
}

summary_parts.append('<h3>Excluded Treatments (Causal-Context Contamination)</h3>')
summary_parts.append('<ul>')
causal_in_data = filter_df[filter_df['Category'] == 'Causal-context']
for _, row in causal_in_data.iterrows():
    reason = causal_reasons.get(row['Treatment'].lower(), 'procedure or causal context, not recovery')
    summary_parts.append(
        f'<li><strong>{row["Treatment"].title()}</strong> ({row["% Positive"]:.0f}% positive, n={row["Users"]}) '
        f'-- excluded: {reason}</li>'
    )
summary_parts.append('</ul>')

summary_parts.append('<h3>Caveats</h3>')
summary_parts.append('<ul>')
summary_parts.append('<li><strong>Self-selected sample</strong> -- users who post about recovery are not representative of all abortion patients</li>')
summary_parts.append('<li><strong>Sentiment is subjective</strong> -- the LLM classifier assigns sentiment labels to user text, which may not capture nuance</li>')
summary_parts.append('<li><strong>Small samples for many treatments</strong> -- statistical power is limited for treatments with fewer than 15 users</li>')
summary_parts.append('<li><strong>No dosage or timing data</strong> -- we know what users mentioned, not how much or when they took it</li>')
summary_parts.append('<li><strong>Not medical advice</strong> -- discuss any treatment decisions with a healthcare provider</li>')
summary_parts.append('</ul>')

summary_parts.append('<h3>Suggested Next Steps</h3>')
summary_parts.append('<ul>')
summary_parts.append('<li>Expand the data window to 3-6 months for larger per-treatment sample sizes</li>')
summary_parts.append('<li>Run demographic-stratified analysis when user profile data is available</li>')
summary_parts.append('<li>Cross-reference top treatments with clinical literature for mechanistic plausibility</li>')
summary_parts.append('</ul>')

summary_parts.append(f'<hr><p><em>{REPORTING_BIAS_DISCLAIMER}</em></p>')

display(HTML('\n'.join(summary_parts)))

In [ ]:
conn.close()